# Clase 2 — Contenedores y Docker

**Objetivo:** entender imágenes, contenedores, volúmenes, redes y Docker Compose, y leer con soltura
el `docker-compose.yaml` del TP, que levanta toda la "empresa" *ML Models and something more Inc.*:
Airflow, MLflow, MinIO (S3), PostgreSQL, Valkey y FastAPI.

> Esta clase funciona **sin Docker instalado**: analiza los archivos del repo. Al final están los
> pasos para instalar Docker Desktop en Windows y levantar todo.

In [1]:
from helpers import ROOT, check, quiz, safe_check

import shutil
import subprocess

import pandas as pd
import yaml
from ipywidgets import interact, widgets

docker = shutil.which("docker")
print("Docker instalado:", docker or "NO (ver la sección 6)")

Docker instalado: NO (ver la sección 6)


## 1. ¿Por qué contenedores?
"En mi máquina funciona" es el problema clásico. Un contenedor empaqueta la aplicación **con** su
entorno (sistema operativo base, Python, librerías) y corre igual en cualquier lado.

| Concepto | Analogía | En el TP |
|---|---|---|
| **Imagen** | la receta / el molde (solo lectura) | `apache/airflow:3.0.2`, `postgres:13` |
| **Contenedor** | la torta: una imagen en ejecución | `mlflow`, `postgres`, `minio` |
| **Dockerfile** | los pasos para construir una imagen | `dockerfiles/airflow/Dockerfile` |
| **Volumen** | disco que sobrevive al contenedor | `db_data`, `minio_data`, `./airflow/dags` |
| **Red** | LAN virtual entre contenedores | `frontend`, `backend` |
| **Compose** | describe y levanta varios contenedores juntos | `docker-compose.yaml` |

A diferencia de una máquina virtual, un contenedor **comparte el kernel** del sistema anfitrión:
arranca en segundos y pesa mucho menos.

In [2]:
quiz(
    "Si borro un contenedor de PostgreSQL, ¿se pierden los datos?",
    ["Siempre", "No, si los datos están en un volumen", "Solo si es de Airflow"],
    answer=1,
    explanation="Por eso el compose define el volumen `db_data`: los datos viven fuera del contenedor. "
    "`docker compose down --volumes` sí los borra.",
)

**Si borro un contenedor de PostgreSQL, ¿se pierden los datos?**

ToggleButtons(options=('Siempre', 'No, si los datos están en un volumen', 'Solo si es de Airflow'), style=Togg…

Output()

## 2. Anatomía de un Dockerfile
Este es el de nuestra imagen de Airflow. Cada instrucción genera una **capa**; Docker reutiliza las
capas que no cambiaron (por eso conviene copiar `requirements.txt` e instalar *antes* de copiar el
código, que cambia más seguido).

In [3]:
print((ROOT / "dockerfiles" / "airflow" / "Dockerfile").read_text(encoding="utf-8"))
print("--- requirements.txt ---")
print((ROOT / "dockerfiles" / "airflow" / "requirements.txt").read_text(encoding="utf-8"))

FROM apache/airflow:3.0.2
COPY requirements.txt /requirements.txt

USER root
RUN apt-get update \
  && apt-get install -y --no-install-recommends \
         git \
  && apt-get autoremove -yqq --purge \
  && apt-get clean \
  && rm -rf /var/lib/apt/lists/*

USER airflow
COPY requirements.txt /requirements.txt

RUN pip install --upgrade pip
RUN pip install --no-cache-dir apache-airflow[virtualenv]==3.0.2 -r /requirements.txt

--- requirements.txt ---
# Mismas versiones que el entorno local (requirements.txt): el modelo que se registra en
# MLflow desde un notebook tiene que poder cargarse en las tareas de Airflow y viceversa.
mlflow==3.18.0
boto3==1.44.0
scikit-learn==1.9.1
xgboost==3.4.1



| Instrucción | Qué hace |
|---|---|
| `FROM` | imagen base de la que partimos |
| `COPY` | copia archivos del contexto de build a la imagen |
| `RUN` | ejecuta un comando **al construir** (instalar paquetes) |
| `USER` | usuario con el que corren las instrucciones siguientes |
| `EXPOSE` | documenta el puerto que usa la app |
| `CMD` / `ENTRYPOINT` | comando que se ejecuta **al arrancar** el contenedor |

### ✏️ Ejercicio 1 — escribí un Dockerfile
Escribí (como texto) un Dockerfile para una API de FastAPI que esté en `app.py`, con sus
dependencias en `requirements.txt`, que escuche en el puerto 8800. Tiene que tener `FROM`, copiar e
instalar `requirements.txt` con `pip`, copiar el código, `EXPOSE 8800` y un `CMD` que corra `uvicorn`.

In [4]:
mi_dockerfile = """
FROM ...
"""  # ← completá

In [5]:
def _revisar_dockerfile(texto: str) -> bool:
    lineas = [line.strip() for line in texto.strip().splitlines() if line.strip()]
    instr = [line.split()[0].upper() for line in lineas]
    pip_idx = next(i for i, line in enumerate(lineas) if line.upper().startswith("RUN") and "pip" in line)
    app_idx = next(i for i, line in enumerate(lineas) if line.upper().startswith("COPY") and "requirements" not in line)
    return (
        instr[0] == "FROM" and "..." not in lineas[0]
        and any(line.upper().startswith("COPY") and "requirements" in line for line in lineas)
        and pip_idx < app_idx  # instalar antes de copiar el código: aprovecha la caché de capas
        and any(line.upper() == "EXPOSE 8800" for line in lineas)
        and any(line.upper().startswith("CMD") and "uvicorn" in line for line in lineas)
    )


safe_check(
    "Ejercicio 1",
    lambda: _revisar_dockerfile(mi_dockerfile),
    "FROM python:3.12-slim → COPY requirements.txt . → RUN pip install -r requirements.txt → "
    "COPY . . → EXPOSE 8800 → CMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8800\"]",
)

⏳ **Ejercicio 1**: todavía no. *Pista:* FROM python:3.12-slim → COPY requirements.txt . → RUN pip install -r requirements.txt → COPY . . → EXPOSE 8800 → CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8800"] (error: `StopIteration: `)

False

<details><summary>Ver solución</summary>

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8800
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8800"]
```
`--host 0.0.0.0` es clave: con `127.0.0.1` la API solo aceptaría conexiones desde *dentro* del
contenedor.
</details>

## 3. El docker-compose del TP
Leemos el YAML real y armamos una tabla de servicios:

In [6]:
compose = yaml.safe_load((ROOT / "docker-compose.yaml").read_text(encoding="utf-8"))
env = dict(
    line.split("=", 1) for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines()
    if "=" in line and not line.startswith("#")
)


def resolver(texto: str) -> str:
    """Reemplaza ${VAR:-default} usando el archivo .env (como hace docker compose)."""
    import re

    def sub(m):
        var, _, default = m.group(1).partition(":-")
        return env.get(var, default)

    return re.sub(r"\$\{([^}]+)\}", sub, str(texto))


filas = []
for nombre, svc in compose["services"].items():
    filas.append({
        "servicio": nombre,
        "imagen / build": svc.get("image") or svc.get("build"),
        "puertos (host:contenedor)": ", ".join(resolver(p) for p in svc.get("ports", [])),
        "perfiles": ", ".join(svc.get("profiles", [])),
        "depende de": ", ".join(svc.get("depends_on", {}) if isinstance(svc.get("depends_on"), dict)
                                 else svc.get("depends_on", [])),
    })
servicios = pd.DataFrame(filas).set_index("servicio")
servicios

,imagen / build,puertos (host:contenedor),perfiles,depende de
servicio,,,,
postgres,postgres_system,5432:5432,"airflow, mlflow, all",
redis,valkey/valkey:8.1-bookworm,,,
s3,coollabsio/minio:RELEASE.2025-04-22T22-12-26Z,"9000:9000, 9001:9001","mlflow, airflow, all",
create_s3_buckets,coollabsio/minio:RELEASE.2025-04-22T22-12-26Z,,"mlflow, airflow, all",s3
mlflow,mlflow,5001:5000,"mlflow, all",postgres
fastapi,backend_fastapi,8800:8800,all,"mlflow, airflow-apiserver"
airflow-apiserver,${AIRFLOW_IMAGE_NAME:-extending_airflow:latest},8080:8080,"airflow, all","postgres, redis, airflow-init"
airflow-scheduler,${AIRFLOW_IMAGE_NAME:-extending_airflow:latest},,"airflow, all","postgres, redis, airflow-init"
airflow-dag-processor,${AIRFLOW_IMAGE_NAME:-extending_airflow:latest},,"airflow, all","postgres, redis, airflow-init"


### 🎛️ Inspeccioná un servicio

In [7]:
@interact(servicio=widgets.Dropdown(options=list(compose["services"]), value="mlflow"))
def inspeccionar(servicio):
    svc = dict(compose["services"][servicio])
    svc.pop("<<", None)
    print(yaml.safe_dump(svc, sort_keys=False, allow_unicode=True))

interactive(children=(Dropdown(description='servicio', index=4, options=('postgres', 'redis', 's3', 'create_s3…

Cosas para notar:

- **`x-airflow-common`** es un *anchor* de YAML (`&airflow-common`): todos los servicios de Airflow
  lo reutilizan con `<<: *airflow-common`. Evita repetir la misma configuración seis veces (DRY otra vez).
- **Nuestros cambios** respecto del repo base de la cátedra: montamos `./src` (el paquete `burnout`) y
  `./data` en Airflow, y definimos `PYTHONPATH` y `MLFLOW_TRACKING_URI`.
- **`depends_on` con `condition: service_healthy`**: un servicio espera a que el otro pase su
  *healthcheck*. MLflow no arranca hasta que PostgreSQL responde.

## 4. Puertos y redes: la pregunta que más confunde

In [8]:
quiz(
    "El servicio mlflow tiene ports: \"${MLFLOW_PORT:-5000}:5000\" y en .env MLFLOW_PORT=5001. "
    "¿Desde tu navegador, dónde está la UI de MLflow?",
    ["http://localhost:5000", "http://localhost:5001", "http://mlflow:5000"],
    answer=1,
    explanation="El formato es HOST:CONTENEDOR. En tu máquina se expone el 5001; adentro el proceso escucha en 5000.",
)

**El servicio mlflow tiene ports: "${MLFLOW_PORT:-5000}:5000" y en .env MLFLOW_PORT=5001. ¿Desde tu navegador, dónde está la UI de MLflow?**

ToggleButtons(options=('http://localhost:5000', 'http://localhost:5001', 'http://mlflow:5000'), style=ToggleBu…

Output()

In [9]:
quiz(
    "Una tarea de Airflow (dentro de Docker) quiere hablar con MLflow. ¿Qué URI usa?",
    ["http://localhost:5001", "http://mlflow:5000", "http://127.0.0.1:5000"],
    answer=1,
    explanation="Dentro de la red de Docker, cada servicio se resuelve por su nombre y se usa el puerto "
    "del contenedor. `localhost` dentro de un contenedor es el propio contenedor. Por eso en el compose "
    "pusimos MLFLOW_TRACKING_URI=http://mlflow:5000 y AWS_ENDPOINT_URL_S3=http://s3:9000.",
)

**Una tarea de Airflow (dentro de Docker) quiere hablar con MLflow. ¿Qué URI usa?**

ToggleButtons(options=('http://localhost:5001', 'http://mlflow:5000', 'http://127.0.0.1:5000'), style=ToggleBu…

Output()

### ✏️ Ejercicio 2 — completá la tabla de URLs
Con la tabla de servicios y el `.env`, completá el diccionario con la URL que usarías **desde tu
navegador** para cada servicio.

In [10]:
urls = {
    "airflow": "http://localhost:____",
    "mlflow": "http://localhost:____",
    "minio_consola": "http://localhost:____",
    "fastapi_docs": "http://localhost:____/docs",
}  # ← completá

In [11]:
esperadas = {
    "airflow": f"http://localhost:{env['AIRFLOW_PORT']}",
    "mlflow": f"http://localhost:{env['MLFLOW_PORT']}",
    "minio_consola": f"http://localhost:{env['MINIO_PORT_UI']}",
    "fastapi_docs": f"http://localhost:{env['FASTAPI_PORT']}/docs",
}
check("Ejercicio 2", urls == esperadas, "mirá las variables *_PORT del archivo .env (y MINIO_PORT_UI para la consola).")

⏳ **Ejercicio 2**: todavía no. *Pista:* mirá las variables *_PORT del archivo .env (y MINIO_PORT_UI para la consola).

False

<details><summary>Ver solución</summary>

Airflow 8080 · MLflow 5001 · consola de MinIO 9001 · docs de la API 8800/docs.
</details>

## 5. Perfiles
`docker compose --profile all up` levanta todo. Con `--profile mlflow` solo lo necesario para MLflow
(PostgreSQL, MinIO y MLflow): útil para el notebook del experimento sin gastar RAM en Airflow.

In [12]:
perfiles = {}
for nombre, svc in compose["services"].items():
    for p in svc.get("profiles", ["(siempre)"]):
        perfiles.setdefault(p, []).append(nombre)
for p, svcs in perfiles.items():
    print(f"{p:>10}: {', '.join(svcs)}")

   airflow: postgres, s3, create_s3_buckets, airflow-apiserver, airflow-scheduler, airflow-dag-processor, airflow-worker, airflow-triggerer, airflow-init
    mlflow: postgres, s3, create_s3_buckets, mlflow
       all: postgres, s3, create_s3_buckets, mlflow, fastapi, airflow-apiserver, airflow-scheduler, airflow-dag-processor, airflow-worker, airflow-triggerer, airflow-init
 (siempre): redis
     debug: airflow-cli


## 6. Instalar Docker y levantar el entorno (Windows)
1. **Habilitar la virtualización** en la BIOS/UEFI si está deshabilitada (Intel VT-x / AMD-V).
2. En PowerShell **como administrador**: `wsl --install` y reiniciar.
3. Instalar [Docker Desktop](https://www.docker.com/products/docker-desktop/) con el backend WSL 2.
4. En la raíz del repo:

```bash
docker compose --profile all up -d     # primera vez: construye las imágenes (varios minutos)
docker ps                              # todos deberían quedar "healthy"
docker compose --profile all down      # apagar (los datos quedan en los volúmenes)
docker compose down --rmi all --volumes  # borrar TODO, incluidos datos
```

Comandos útiles para depurar:

```bash
docker compose logs -f mlflow          # ver logs de un servicio
docker exec -it mlflow bash            # entrar a un contenedor
docker compose build airflow-apiserver # reconstruir tras cambiar requirements
```

In [13]:
if docker:
    print(subprocess.run([docker, "ps", "--format", "table {{.Names}}\t{{.Status}}"],
                         capture_output=True, text=True).stdout or "(sin contenedores corriendo)")
else:
    print("Docker todavía no está instalado: seguí los pasos de arriba.")

Docker todavía no está instalado: seguí los pasos de arriba.


## Resumen
- Imagen = molde; contenedor = imagen corriendo; volumen = datos que sobreviven; red = DNS por nombre.
- Puertos `HOST:CONTENEDOR`; entre contenedores se usa `nombre_servicio:puerto_contenedor`.
- Nuestro compose extiende el de la cátedra para montar el paquete `burnout` en Airflow.

Siguiente: **C3 — MLflow**.